# Delta Lake maintenance on minilake: small files, `OPTIMIZE`, `VACUUM`

Every micro-batch pipeline eventually creates the same problem: thousands of tiny Parquet
files, one per commit per partition. Reads slow down, the transaction log grows, and
storage fills with data nobody reads. Delta Lake's answer is `OPTIMIZE` — bin-pack the
small files into large ones — followed by `VACUUM`, which physically deletes the files
nothing points at any more.

This notebook runs that whole cycle for real, and watches it from both sides:

- **Spark** — real `OPTIMIZE` / `VACUUM` / `DESCRIBE HISTORY` statements, the same ones you
  would run on Databricks, submitted to minilake's Jobs API and executed by `spark-submit`
  in a sibling Spark container.
- **minilake** — the same table, reached through the unmodified `databricks-sdk`: Unity
  Catalog registration, a SQL warehouse, and the SQL Statement Execution API.

There is no sync step between the two. One directory of Delta files, two clients.

> **This kernel has no `pyspark`, and does not need one.** That is deliberate: on Databricks
> your notebook runs against a cluster, not inside the control plane. Every Spark statement
> below travels through the Jobs API. `pip install delta` installs an unrelated PyPI package
> and will not help — nothing here imports `delta`.

## 1. Connect to minilake

`MINILAKE_HOST` and `MINILAKE_DATA_DIR` are injected into the kernel by minilake when it
starts JupyterLab; the defaults are what you would use running this notebook anywhere else.

Reset first, so the notebook can be re-run from scratch at any time. The constants are the
knobs: `BATCHES` is how many micro-batch commits to make, and it is what controls how bad
the small-files problem gets.

In [ ]:
import os
import re
import shutil
import statistics
import time

import requests
from databricks.sdk import WorkspaceClient

# minilake injects MINILAKE_HOST into the kernel as a full URL. Outside that kernel the
# same name is the server's *bind address* (0.0.0.0 in the image), so only take it when it
# actually looks like a URL.
_injected = os.environ.get("MINILAKE_HOST", "")
MINILAKE = _injected if _injected.startswith("http") else "http://localhost:8000"
DATA_DIR = os.environ.get("MINILAKE_DATA_DIR", "/data")

# Tunables. Raise BATCHES if the effect of compaction looks flat on your machine.
BATCHES = 50          # one micro-batch append per iteration -> one Delta commit each
ROWS_PER_BATCH = 200
REGIONS = ["us-east", "us-west", "eu-central", "ap-south"]

CATALOG, SCHEMA, TABLE = "delta_ops", "retail", "sales"
FULL_NAME = f"{CATALOG}.{SCHEMA}.{TABLE}"
# On the data volume: the Spark container that writes these files and the minilake that
# reads them mount it at the same path, which is why one storage location works for both.
TABLE_PATH = f"{DATA_DIR}/delta/{CATALOG}/{SCHEMA}/{TABLE}"

requests.post(f"{MINILAKE}/_minilake/reset").raise_for_status()
shutil.rmtree(TABLE_PATH, ignore_errors=True)

w = WorkspaceClient(host=MINILAKE, token="dev")
w.catalogs.create(name=CATALOG, comment="Delta maintenance demo")
w.schemas.create(name=SCHEMA, catalog_name=CATALOG)
wh = w.warehouses.create(name="ops_wh")


def sql(statement):
    """Run SQL through minilake's Databricks SQL Statement Execution API."""
    r = w.statement_execution.execute_statement(warehouse_id=wh.id, statement=statement)
    if r.status.state.value != "SUCCEEDED":
        raise RuntimeError(f"{r.status.state}: {r.status.error}")
    return r.result.data_array or []


print(f"connected to {MINILAKE} as {w.current_user.me().user_name}")
print(f"catalog {CATALOG}.{SCHEMA} + warehouse {wh.id} ready")

## 2. A helper that runs Spark for real

Stage the script in the workspace, create a one-off job with a `spark_python_task`, run it,
wait, hand back its output. This is exactly the `run_python_script` recipe from the
quickstart notebook, with two additions: every script gets the same Delta-configured
session prepended, and the Spark log noise is filtered out of what gets printed.

The `libraries` entry is what puts the Delta jars on the classpath — the base Spark image
ships none, and without it `format("delta")` fails with `DATA_SOURCE_NOT_FOUND`.

In [ ]:
import base64
import uuid

DELTA_PACKAGE = "io.delta:delta-spark_2.12:3.2.1"
BEGIN, END = "--8<-- begin", "--8<-- end"

# Prepended to every script below. Two settings beyond the usual Delta wiring:
#   retentionDurationCheck   -- Spark refuses `VACUUM ... RETAIN 0 HOURS` without it.
#                               That check exists for a good reason; section 10 gets into it.
#   parallelPartitionDiscovery -- VACUUM lists the table tree with this parallelism, and the
#                               10000 default means 10000 near-empty tasks on a table this size.
PRELUDE = f"""
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder.appName("minilake-delta-maintenance")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.databricks.delta.retentionDurationCheck.enabled", "false")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.sources.parallelPartitionDiscovery.parallelism", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

TABLE_PATH = "{TABLE_PATH}"
print("{BEGIN}")
"""

# spark-submit interleaves its own logging, and the JVM its stack traces, with whatever
# the script prints. Drop both -- a failing run prints its raw tail below anyway.
_NOISE = re.compile(
    r"^(\d\d/\d\d/\d\d \d\d:\d\d:\d\d |\s+at |Caused by: |\s*\.\.\. \d+ more"
    r"|[\w.$]+(Exception|Error)[:(]|It is possible the underlying files)"
)


def run_spark(body, name="step", timeout=900, raw=False):
    """Run a PySpark snippet in a sibling Spark container and print what it printed."""
    script = PRELUDE + body + f'\nprint("{END}")\n'
    tag = f"{name}-{uuid.uuid4().hex[:8]}"
    path = f"/Shared/notebook/{tag}.py"

    requests.post(
        f"{MINILAKE}/api/2.0/workspace/import",
        json={
            "path": path,
            "content": base64.b64encode(script.encode()).decode(),
            "language": "PYTHON",
            "format": "SOURCE",
            "overwrite": True,
        },
    ).raise_for_status()

    job_id = requests.post(
        f"{MINILAKE}/api/2.2/jobs/create",
        json={
            "name": tag,
            "tasks": [
                {
                    "task_key": "main",
                    "spark_python_task": {"python_file": path},
                    "libraries": [{"maven": {"coordinates": DELTA_PACKAGE}}],
                }
            ],
        },
    ).json()["job_id"]
    run_id = requests.post(
        f"{MINILAKE}/api/2.2/jobs/run-now", json={"job_id": job_id}
    ).json()["run_id"]

    started = time.time()
    while time.time() - started < timeout:
        run = requests.get(
            f"{MINILAKE}/api/2.2/jobs/runs/get", params={"run_id": run_id}
        ).json()
        if (run.get("state") or {}).get("life_cycle_state") == "TERMINATED":
            break
        time.sleep(2)
    else:
        raise TimeoutError(f"run {run_id} did not finish in {timeout}s")

    out = requests.get(
        f"{MINILAKE}/api/2.2/jobs/runs/get-output", params={"run_id": run_id}
    ).json()
    logs = out.get("logs") or ""
    state = (run.get("state") or {}).get("result_state")

    if state != "SUCCESS" or raw:
        print(out.get("error") or "")
        print("\n".join(logs.splitlines()[-40:]))
        if state != "SUCCESS":
            raise RuntimeError(f"{name}: {state}")

    # Everything the script itself printed, minus the noise interleaved with it.
    body_out = logs.split(BEGIN, 1)[-1].split(END, 1)[0]
    text = "\n".join(l for l in body_out.splitlines() if not _NOISE.match(l))
    print(f"[run {run_id}: {state} in {time.time() - started:.0f}s]\n")
    print(text.strip("\n"))

## 3. Register the table before writing to it

minilake stores only the *metadata* for an EXTERNAL table — the data is whatever real Delta
files live at `storage_location`. Nothing is written here.

Register it first, though, not after. Creating an EXTERNAL Delta table is what creates the
storage location, and minilake makes that directory world-writable on the way — because the
process that writes the data (a job container, your own Spark, some other tooling) runs
under a different UID than the server. Write first and Spark has nowhere to put the files.

In [ ]:
from databricks.sdk.service.catalog import DataSourceFormat, TableType

w.tables.create(
    name=TABLE,
    catalog_name=CATALOG,
    schema_name=SCHEMA,
    table_type=TableType.EXTERNAL,
    data_source_format=DataSourceFormat.DELTA,
    storage_location=TABLE_PATH,
)

info = w.tables.get(full_name=FULL_NAME)
print(f"{info.full_name}  ({info.table_type.value}, {info.data_source_format.value})")
print(f"  location: {info.storage_location}")
print(f"  columns : {len(info.columns or [])}  (nothing written yet)")

## 4. Manufacture the small-files problem

The realistic way to end up with thousands of small files is not one badly tuned write — it
is thousands of *correctly* tuned small writes. So this is a loop of `BATCHES` separate
appends, each one its own commit, exactly like a streaming sink with a short trigger
interval. One Spark session does all of them, the way a long-running stream would.

The table is partitioned by `region`, deliberately: partition columns are the only thing
`OPTIMIZE ... WHERE` accepts as a predicate, which section 9 uses.

In [ ]:
run_spark(f'''
import random
from datetime import datetime, timedelta

random.seed(7)
epoch = datetime(2026, 1, 1)
REGIONS = {REGIONS!r}

for batch_id in range({BATCHES}):
    rows = [
        (
            batch_id * {ROWS_PER_BATCH} + i,
            random.randint(1, 500),
            REGIONS[i % len(REGIONS)],
            round(random.uniform(5.0, 900.0), 2),
            epoch + timedelta(minutes=batch_id * 15 + i),
        )
        for i in range({ROWS_PER_BATCH})
    ]
    df = spark.createDataFrame(rows, ["id", "customer_id", "region", "amount", "event_ts"])
    (
        df.repartition(len(REGIONS), "region")
        .write.format("delta")
        .mode("append")
        .partitionBy("region")
        .save(TABLE_PATH)
    )

print("{BATCHES} micro-batch commits written to", TABLE_PATH)
''', name="micro-batches")

Ask Unity Catalog for the table again. We never declared a schema — the columns come back
anyway, read out of the Delta log that Spark just wrote.

In [ ]:
info = w.tables.get(full_name=FULL_NAME)
for c in info.columns:
    print(f"  {c.name}: {c.type_text}")

## 5. Measure the damage — through minilake's API

Everything below goes over HTTP to minilake. Two views of the table:

- **On disk** — `glob()` over the data files. Delta names data files `part-*.parquet`,
  which conveniently excludes the checkpoint files inside `_delta_log/`.
- **Live** — the transaction log itself, replayed in SQL: every `add.path`, minus every
  `remove.path`. That difference is precisely the set of files a reader will open.

Right now the two agree. After `OPTIMIZE` they will not, and the gap is the whole story.

In [ ]:
DATA_GLOB = f"{TABLE_PATH}/**/part-*.parquet"
LOG_GLOB = f"{TABLE_PATH}/_delta_log/*.json"

# Forcing the schema means `remove` is a real (all-null) column even before anything has
# been removed, so the same query works at every stage of the notebook.
_LOG = (
    f"read_json('{LOG_GLOB}', format='newline_delimited', filename=true, "
    "columns={'add': 'STRUCT(\"path\" VARCHAR, \"size\" BIGINT)', "
    "'remove': 'STRUCT(\"path\" VARCHAR)'})"
)


def layout():
    """What is physically on disk, and what the transaction log says is live."""
    files = sql(f"SELECT count(*) FROM glob('{DATA_GLOB}')")[0][0]
    # Rows as a reader sees them: delta_scan() opens only the live files.
    rows = sql(f"SELECT count(*) FROM {FULL_NAME}")[0][0]

    commits, adds, removes = sql(f"""
        SELECT count(DISTINCT filename), count(add), count(remove) FROM {_LOG}
    """)[0]

    # Delta log replay, in SQL: a file is live unless some later commit tombstoned it.
    live_files, live_bytes, dead_files, dead_bytes = sql(f"""
        SELECT
            coalesce(count(*)  FILTER (WHERE NOT dead), 0),
            coalesce(sum(size) FILTER (WHERE NOT dead), 0),
            coalesce(count(*)  FILTER (WHERE dead), 0),
            coalesce(sum(size) FILTER (WHERE dead), 0)
        FROM (
            SELECT add.path AS path,
                   add.size AS size,
                   add.path IN (SELECT remove.path FROM {_LOG} WHERE remove IS NOT NULL) AS dead
            FROM {_LOG} WHERE add IS NOT NULL
        )
    """)[0]

    return {
        "files_on_disk": int(files),
        "rows": int(rows),
        "rows_per_live_file": int(rows) / max(int(live_files), 1),
        "commits": int(commits),
        "adds": int(adds),
        "removes": int(removes),
        "live_files": int(live_files),
        "live_bytes": int(live_bytes),
        "dead_files": int(dead_files),
        "dead_bytes": int(dead_bytes),
    }


def kib(n):
    return f"{n / 1024:,.0f} KiB"


def show_layout(title):
    s = layout()
    stale_on_disk = s["files_on_disk"] - s["live_files"]
    print(title)
    print(f"  parquet files on disk       {s['files_on_disk']:>6}")
    print(f"    live, readers open these  {s['live_files']:>6}   {kib(s['live_bytes'])}")
    print(f"    dead, not yet deleted     {stale_on_disk:>6}   {kib(s['dead_bytes']) if stale_on_disk else '0 KiB'}")
    print(f"  rows per live file          {s['rows_per_live_file']:>6,.0f}")
    print(f"  log: {s['commits']} commits, {s['adds']} adds, {s['removes']} removes")
    return s


before = show_layout("After the micro-batch run:")

assert before["commits"] == BATCHES, before
assert before["files_on_disk"] == before["live_files"], "no maintenance has run yet"
assert before["files_on_disk"] >= 150, "not enough small files to make the point"

## 6. Benchmark the read path

The same statement, through minilake's SQL Statement Execution API. One warm-up call to pay
the metadata cost, then the median of five.

The result rows are kept as the correctness invariant: compaction and vacuuming are not
allowed to change a single number, and that gets re-checked after every step.

In [ ]:
AGG = f"""
    SELECT region, count(*) AS orders, round(sum(amount), 2) AS revenue
    FROM {FULL_NAME}
    GROUP BY region
    ORDER BY region
"""


def bench(statement, reps=5):
    sql(statement)  # warm-up
    timings = []
    for _ in range(reps):
        t0 = time.perf_counter()
        rows = sql(statement)
        timings.append((time.perf_counter() - t0) * 1000)
    return statistics.median(timings), rows


before_ms, expected_rows = bench(AGG)
print(f"median {before_ms:.0f} ms over {before['live_files']} files\n")
for region, orders, revenue in expected_rows:
    print(f"  {region:<12} {orders:>6} orders  {revenue:>12}")

## 7. `DESCRIBE HISTORY` and time travel

Each of those appends is a numbered version. `DESCRIBE HISTORY` and `VERSION AS OF` are
Delta *protocol* features, so they run in Spark — minilake's SQL endpoint speaks DuckDB and
implements neither. What it can do is read the raw transaction log, which is exactly what
section 5 did.

Same log, two readers.

In [ ]:
run_spark(r'''
history = spark.sql("DESCRIBE HISTORY delta.`" + TABLE_PATH + "`")
print(history.count(), "versions; most recent first:")
history.select("version", "operation", "operationMetrics").show(5, truncate=60)

latest = history.selectExpr("max(version)").first()[0]
v0 = spark.read.format("delta").option("versionAsOf", 0).load(TABLE_PATH).count()
now = spark.read.format("delta").load(TABLE_PATH).count()
print("version 0 has", v0, "rows; version", latest, "has", now)
''', name="history")

## 8. `OPTIMIZE` — bin-pack the small files

The statement is the Databricks one, unchanged. Watch `numFilesRemoved` against
`numFilesAdded`.

In [ ]:
run_spark(r'''
metrics = spark.sql("OPTIMIZE delta.`" + TABLE_PATH + "`").select("metrics.*").first().asDict()
for key in ("numFilesAdded", "numFilesRemoved", "partitionsOptimized", "numBatches"):
    if key in metrics:
        print(" ", key.ljust(22), metrics[key])
''', name="optimize")

### What actually changed on disk

This is the part that surprises people. Count the files again:

In [ ]:
optimized = show_layout("After OPTIMIZE:")

assert optimized["live_files"] < before["live_files"], optimized
assert optimized["files_on_disk"] > optimized["live_files"], "tombstones should still exist"
print(
    f"\nlive files: {before['live_files']} -> {optimized['live_files']}"
    f"   |   still on disk: {optimized['files_on_disk']}"
)

**The number of Parquet files on disk went *up*, not down.**

`OPTIMIZE` wrote new compacted files and appended `remove` actions for the old ones. A
`remove` is a tombstone — a note in the log saying readers should stop opening this file.
It deletes nothing. That is what keeps version 0 readable, and it is why `OPTIMIZE` on its
own never reclaims a byte of storage.

`VACUUM`, in section 10, is the half that does.

## 9. Re-benchmark, and check nothing moved

Fewer, larger files means fewer file opens, fewer footers to parse, less metadata to
reconcile. How much that is worth depends on the storage and the data size. The invariant
that must hold regardless is that the answers are identical.

In [ ]:
after_ms, after_rows = bench(AGG)

assert after_rows == expected_rows, "OPTIMIZE must not change a single value"

print(f"before : {before_ms:6.0f} ms  over {before['live_files']:>3} files")
print(f"after  : {after_ms:6.0f} ms  over {optimized['live_files']:>3} files")
print(f"ratio  : {before_ms / after_ms:.2f}x")
print("\nresults identical, verified row by row")

### `ZORDER` and partition-scoped `OPTIMIZE`

`OPTIMIZE ... WHERE` accepts partition predicates, so maintenance can run one partition at a
time instead of rewriting the whole table. `ZORDER BY` additionally co-locates rows with
similar values in the same files, so min/max statistics can skip more of them.

Be clear-eyed about what this proves here. The Databricks statement runs unmodified and the
table stays readable through the SDK — that is the compatibility claim, and it is the one
being demonstrated. The *read* is DuckDB's `delta_scan`, which does its own Parquet
row-group pruning, so at four files and ten thousand rows any timing difference is mostly
noise. Z-ordering pays off at a scale this notebook deliberately does not reach.

In [ ]:
run_spark(r'''
statement = "OPTIMIZE delta.`" + TABLE_PATH + "` WHERE region = 'us-east' ZORDER BY (customer_id)"
metrics = spark.sql(statement).select("metrics.*").first().asDict()
print("  numFilesAdded   ", metrics.get("numFilesAdded"))
print("  numFilesRemoved ", metrics.get("numFilesRemoved"))
print("  zOrderStats     ", metrics.get("zOrderStats"))
''', name="zorder")

In [ ]:
point_lookup = f"""
    SELECT count(*) AS hits, round(sum(amount), 2) AS revenue
    FROM {FULL_NAME}
    WHERE region = 'us-east' AND customer_id = 42
"""
lookup_ms, lookup_rows = bench(point_lookup)
print(f"\npoint lookup through minilake: {lookup_rows[0]} in {lookup_ms:.0f} ms")

## 10. `VACUUM` — reclaim the tombstoned files

`RETAIN 0 HOURS` is a demo-only setting, and Spark refuses it unless the retention check is
disabled, which the session prelude in section 2 did.

The default is **7 days**, and the reason is concurrency rather than caution for its own
sake: a query that started before the `OPTIMIZE` is still reading the old files. Vacuum them
out from under it and it fails. The retention window is how long you promise not to do that.

`DRY RUN` first, which is how this should always be approached in production.

In [ ]:
run_spark(r'''
condemned = spark.sql("VACUUM delta.`" + TABLE_PATH + "` RETAIN 0 HOURS DRY RUN")
print(condemned.count(), "files would be deleted, e.g.:")
condemned.show(3, truncate=80)
''', name="vacuum-dry-run")

pre_vacuum = layout()

In [ ]:
run_spark(r'''
spark.sql("VACUUM delta.`" + TABLE_PATH + "` RETAIN 0 HOURS")
print("vacuum complete")

# The same time-travel read that worked in section 7. Note the aggregate: .count() alone
# would still succeed, because Delta answers it from row counts in the log without opening
# a single Parquet file.
from pyspark.sql import functions as F

try:
    total = (
        spark.read.format("delta").option("versionAsOf", 0).load(TABLE_PATH)
        .agg(F.sum("amount")).first()[0]
    )
    print("version 0 still readable, sum(amount) =", total)
except Exception as e:
    # Py4J wraps the JVM stack trace into one string; the useful line is buried in it.
    import re

    found = re.search(r"(File file:\S+ does not exist)", str(e))
    print("version 0 is gone --", found.group(1) if found else str(e).splitlines()[0])
''', name="vacuum")

In [ ]:
vacuumed = show_layout("After VACUUM:")

assert vacuumed["files_on_disk"] == vacuumed["live_files"], vacuumed
print(f"\nreclaimed {kib(pre_vacuum['dead_bytes'])} across {pre_vacuum['files_on_disk'] - pre_vacuum['live_files']} files")
print(f"the log still records {vacuumed['removes']} removes -- the tombstones outlive the bytes")

### The data is untouched. The history is not.

Read the table through minilake again — identical to section 6, byte for byte. What is gone
is version 0, whose files were just deleted: **`VACUUM` is what ends time travel.** That is
the trade being made, and the 7-day default is how long Delta gives you to change your mind.

In [ ]:
final_ms, final_rows = bench(AGG)
assert final_rows == expected_rows, "VACUUM must not change a single value"
print(f"current data still reads fine through minilake ({final_ms:.0f} ms), results identical")
for region, orders, revenue in final_rows:
    print(f"  {region:<12} {orders:>6} orders  {revenue:>12}")

## 11. What ran where

| Operation | Ran on | Why |
|---|---|---|
| `OPTIMIZE`, `OPTIMIZE ... ZORDER BY`, `VACUUM` | Spark + Delta 3.2, via the Jobs API | Delta *protocol* operations: they rewrite files and append log actions. On real Databricks these run in Spark too, not in the SQL warehouse's dialect. |
| `DESCRIBE HISTORY`, `VERSION AS OF` | Spark + Delta 3.2, via the Jobs API | Same reason. |
| catalog / schema / table registration | minilake, via `databricks-sdk` | Unity Catalog API |
| warehouse creation | minilake, via `databricks-sdk` | SQL Warehouses API |
| every `SELECT`, every file count, the log replay | minilake, via `databricks-sdk` | SQL Statement Execution API |

One directory of Delta files underneath all of it, with no synchronisation step anywhere.

**Where minilake stops.** Its SQL endpoint executes DuckDB, not Spark SQL, so `OPTIMIZE`,
`VACUUM`, `DESCRIBE HISTORY` and `VERSION AS OF` are not part of its dialect — send those to
Spark, which is what section 2's helper does. What minilake emulates is the workspace around
the table: the catalog, the warehouse, the statement API, the Jobs API. That is enough to run
real Delta maintenance and observe every effect of it with unmodified Databricks client code.

Every Spark run above is listed under **Jobs** (`/ui/jobs`) with its full logs, and the table
is in the **catalog** (`/ui/catalog`) with the columns Delta reported.

In [ ]:
info = w.tables.get(full_name=FULL_NAME)
final = layout()
print(f"{info.full_name} still registered, {len(info.columns)} columns, at {info.storage_location}")
print(f"{final['live_files']} files hold {final['rows']} rows")